# Model Evaluation Lab

This notebook shows why validation design matters. You will compare a random split with a group held out split using synthetic grouped data.


In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/synthetic_sensor_microbiome.csv"
df = pd.read_csv(DATA_PATH)
df.head()


## Build one model

We include `site` as a categorical feature to make the leakage problem visible. In the random split, each site appears in both training and test data. In the group held out split, at least one site is unseen during training.


In [ ]:
target = "risk_score"
features = [
    "site",
    "temperature_c",
    "humidity",
    "vibration_index",
    "sparse_count_a",
    "sparse_count_b",
]

X_raw = df[features]
y = df[target]
groups = df["site"]

def make_design_matrix(frame, known_sites, numeric_mean, numeric_std):
    numeric_columns = [c for c in features if c != "site"]
    numeric = frame[numeric_columns].astype(float)
    numeric = (numeric - numeric_mean) / numeric_std
    site_columns = {
        f"site_{site}": (frame["site"] == site).astype(float)
        for site in known_sites
    }
    site_df = pd.DataFrame(site_columns, index=frame.index)
    design = pd.concat([numeric, site_df], axis=1)
    design.insert(0, "intercept", 1.0)
    return design

def fit_ridge(X_train, y_train, alpha=1.0):
    Xmat = X_train.to_numpy(float)
    yvec = y_train.to_numpy(float)
    penalty = np.eye(Xmat.shape[1]) * alpha
    penalty[0, 0] = 0.0
    return np.linalg.solve(Xmat.T @ Xmat + penalty, Xmat.T @ yvec)

def predict(X_test, beta):
    return X_test.to_numpy(float) @ beta


In [ ]:
def metrics(y_true, y_pred):
    error = y_true.to_numpy(float) - y_pred
    mae = np.mean(np.abs(error))
    rmse = np.sqrt(np.mean(error ** 2))
    ss_res = np.sum(error ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    return mae, rmse, r2

def evaluate(split_name, train_idx, test_idx):
    train = df.iloc[train_idx]
    test = df.iloc[test_idx]
    known_sites = sorted(train["site"].unique())
    numeric_columns = [c for c in features if c != "site"]
    numeric_mean = train[numeric_columns].astype(float).mean()
    numeric_std = train[numeric_columns].astype(float).std(ddof=0).replace(0, 1)
    X_train = make_design_matrix(train[features], known_sites, numeric_mean, numeric_std)
    X_test = make_design_matrix(test[features], known_sites, numeric_mean, numeric_std)
    beta = fit_ridge(X_train, train[target])
    pred = predict(X_test, beta)
    mae, rmse, r2 = metrics(test[target], pred)
    return {
        "split": split_name,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
    }


## Random split

This split is easy to write, but it may not match the deployment question.


In [ ]:
rng = np.random.default_rng(42)
all_idx = np.arange(len(df))
rng.shuffle(all_idx)
test_size = int(0.25 * len(df))
test_idx = all_idx[:test_size]
train_idx = all_idx[test_size:]

random_result = evaluate("random rows", train_idx, test_idx)
random_result


## Group held out split

This split asks whether the model can generalize to sites that were not represented in training.


In [ ]:
held_out_sites = ["estuary", "harbor"]
test_idx = df.index[df["site"].isin(held_out_sites)].to_numpy()
train_idx = df.index[~df["site"].isin(held_out_sites)].to_numpy()

group_result = evaluate("held out sites", train_idx, test_idx)

print("Held out sites:", sorted(df.iloc[test_idx]["site"].unique()))
group_result


In [ ]:
results = pd.DataFrame([random_result, group_result])
results.round(3)


## Reflection

Write a short explanation:

1. Which split gives the stronger score?
2. Why is that score not necessarily the right estimate for a new site?
3. Which split would you report if the model will be used on future data from an unseen site?
4. What limitation does this synthetic example have?
